# Time Evolution of a spin-1/2
This notebook uses the Lindblad formalism to simulate the time evolution of a spin-1/2 which has been initialized in certain state. It includes:
- Spin Hamiltonian
- Constructing collapse operators
- Time Evolution simulation

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from qutip import (
    tensor, qeye, sigmax, sigmay, sigmaz,
    basis, mesolve, sigmam, Qobj, Bloch
)

from spinfinity import lind, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Spin Hamiltonian

In [ ]:
spin = SpinSys([0.5])
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 0.5]  # T

hamil.calc_EigEnergies(spin, rebuildH=True)
hamil.plot_EigenMatrix(spin);

## Construct collapse operators by hand

In [ ]:
# Define the relaxation time
T1 = 100.0 # ns

# Relaxation Rate
gamma1 = 1.0 / T1

# Define the Lindblad operators
sm = sigmam()
sp = sigmam().dag()

# Calculate the populations of the spin states
bath.calc_Populations(spin)

c_ops = [
    np.sqrt(gamma1*spin.pop_0[1]) * sm, # Relaxation from |1> to |0>
    np.sqrt(gamma1*spin.pop_0[0]) * sp, # Excitation from |0> to |1>
]
# Assign the collapse operators to the SpinSystem
spin.collapse_ops = c_ops

## Time evolution

In [ ]:
# Define an initial state
psi_exc = basis(2, 1) # Excited state |1>

# Define the time array for the simulation
time_array = np.linspace(0, 1000, 1000) # ns

# Calculate the time evolution
result = lind.calc_TimeEvolution(spin, psi_0=psi_exc, time_array=time_array)

# Perform an exponential Fit
x = result.times 
y = result.expect[0]
x_fit, y_fit, popt, perr = hf.fit_exponential(x, y)
A_fit, T_fit, C_fit = popt

# Plot the result
plt.figure(figsize=(7, 4))
plt.plot(x, y, "o", ms=3, label="Simulated trace")
plt.plot(x_fit, y_fit, "-", lw=2, label="Exponential fit")
plt.xlabel(r"Free evolution time $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_z \rangle$")
plt.title(f"Fit: A={A_fit:.4f}, T={T_fit:.2f} ns, C={C_fit:.4f}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Time evolution with thermal initialization

In [ ]:
# Define an initial state
rho_therm = lind.initialize_thermal(spin)

time_array = np.linspace(0, 1000, 1000) # ns
result = lind.calc_TimeEvolution(spin, psi_0=rho_therm, time_array=time_array)
# plot result
lind.plot_TimeEvolution(spin);

## Construct collapse operators of the tunnel junction

In [ ]:
# Tunnel junction parameters
spin.TipPolarization = [0.0,0.0,0.5] 
spin.U = [0] 
spin.G_ss = [2e-9] # Siemens
spin.G_st = 0.2e-9 # Siemens
spin.G_tt = 1
spin.V_DC = 0.1 # mV

bath.calc_Rates(spin)
bath.calc_Populations(spin)
bath.calc_Lifetimes(spin)

# T1 from the tunneling rates
T1_junction = 1/(1/spin.T_state_tot[0] + 1/spin.T_state_tot[1])*1e9 # ns

# calculate the collapse operators and time evolution with the tunneling junction
c_ops = lind.calc_collapseOperators(spin);
spin.collapse_ops = c_ops
time_array = np.linspace(0, 2000, 1000) # ns
result = lind.calc_TimeEvolution(spin, psi_0=psi_exc, time_array=time_array)

# Perform an exponential Fit
x = result.times
y = result.expect[0]
x_fit, y_fit, popt, perr = hf.fit_exponential(x, y)
A_fit, T_fit, C_fit = popt

# Plot the result
print(f"T1 from the tunneling junction: {T1_junction:.2f} ns")
plt.figure(figsize=(7, 4))
plt.plot(x, y, "o", ms=3, label="Simulated trace")
plt.plot(x_fit, y_fit, "-", lw=2, label="Exponential fit")
plt.xlabel(r"time (ns)")
plt.ylabel(r"$\langle S_z \rangle$")
plt.ylim(-1,1)
plt.title(f"Fit: A={A_fit:.4f}, T={T_fit:.2f} ns, C={C_fit:.4f}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Plotting the tunnel current

In [ ]:
lind.plot_TunnelCurrent_time(spin);